# Executor de experimentos

- Usado para a execução de cada experimento no ambiente do Google Colab.
- Cada experimento foi realizado em três rodadas com seeds diferentes
- Os resultados de cada experimento foram salvas em arquivos no Google Drive e num banco de dados

Verificação do ambiente de execução

In [1]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

!nvidia-smi

2.19.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Sat Apr 25 22:00:17 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   36C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                       

Clonagem do repositório atualizado no GitHub

In [2]:
%cd /content
!git clone https://github.com/amartinsmg/classification-of-medical-images-using-cnn.git
%cd /content/classification-of-medical-images-using-cnn

/content
Cloning into 'classification-of-medical-images-using-cnn'...
remote: Enumerating objects: 608, done.
remote: Counting objects: 100% (339/339), done.
remote: Compressing objects: 100% (218/218), done.
remote: Total 608 (delta 181), reused 259 (delta 113), pack-reused 269 (from 1)
Receiving objects: 100% (608/608), 6.82 MiB | 14.64 MiB/s, done.
Resolving deltas: 100% (314/314), done.
/content/classification-of-medical-images-using-cnn


Montagem do Google Drive

In [3]:
from google.colab import drive

drive.mount("/content/drive")
BASE_PATH = "/content/drive/MyDrive/classification-of-medical-images-using-cnn/"

Mounted at /content/drive


Configuração dos parâmetros do experimento

In [4]:
EXPERIMENT_NAME = "example"
BASE_MODEL = "resnet"
NORMALIZATION = "rescaling"
DATA_AUG = True
IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
CLASS_WEIGHT = False
LEARNING_RATE = 0.001
EPOCHS = 10
THRESHOLD = 0.5
seeds = [42, 123, 999]

Criação da engine do banco de dados
- Foi usado um banco de dados sqlite hospedado também no Google Drive

In [5]:
from src.db import insert_run, get_engine

DB_PATH = f"sqlite:///{BASE_PATH}/tests.db"

engine = get_engine(DB_PATH)

Realização das três rodadas do experimento com:
- Treinamento do modelo
- Teste do modelo treinado
- Persitência dos parâmetros e dos resulados da rodada em banco de dados

In [6]:
from src.train import train_pipeline
from src.test import test_pipeline

for i in range(3):
    run_id = i + 1
    config = train_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        data_augmentation=DATA_AUG,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        class_weights=CLASS_WEIGHT,
        learning_rate=LEARNING_RATE,
        epochs=EPOCHS,
        seed=seeds[i],
    )

    metrics = test_pipeline(
        base_dir=BASE_PATH,
        experiment_name=EXPERIMENT_NAME,
        run_id=run_id,
        normalization=NORMALIZATION,
        base_model=BASE_MODEL,
        image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE,
        threshold=THRESHOLD,
    )

    insert_run(
        engine=engine,
        experiment=EXPERIMENT_NAME,
        run_name=f"run_{run_id}",
        config=config,
        metrics=metrics,
    )

Found 5216 files belonging to 2 classes.
Found 16 files belonging to 2 classes.
94765736/94765736 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 879s 5s/step - AUC: 0.7123 - accuracy: 0.7400 - loss: 0.5294 - val_AUC: 0.8047 - val_accuracy: 0.5625 - val_loss: 0.7622
Epoch 2/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 16s 99ms/step - AUC: 0.8533 - accuracy: 0.7828 - loss: 0.4361 - val_AUC: 0.7969 - val_accuracy: 0.6250 - val_loss: 0.7183
Epoch 3/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 16s 97ms/step - AUC: 0.8730 - accuracy: 0.8075 - loss: 0.3970 - val_AUC: 0.7969 - val_accuracy: 0.6250 - val_loss: 0.7484
Epoch 4/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 15s 91ms/step - AUC: 0.8870 - accuracy: 0.8225 - loss: 0.3738 - val_AUC: 0.7969 - val_accuracy: 0.6250 - val_loss: 0.7668
Epoch 5/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 14s 89ms/step - AUC: 0.8961 - accuracy: 0.8330 - loss: 0.3584 - val_AUC: 0.8125 - val_accuracy: 0.6250 - val_loss: 0.7820
Epoch 6/10
163/163 ━━━━━━━━━━━━━━━━━━━━ 15s 90ms/step - AUC: